## 5 Omnilingual Finetuning

This notebook finetunes a model based on Omnilingual.

In [6]:
# %% [markdown]
# # Omnilingual ASR: Romansh Fine-Tuning Pipeline
# Verified against facebookresearch/omnilingual-asr repository structure.

# %%
import os
import io
import yaml
import subprocess
import pandas as pd
import pyarrow as pa
import pyarrow.parquet as pq
import librosa
import soundfile as sf
import numpy as np
import torch
from pathlib import Path
from concurrent.futures import ProcessPoolExecutor, as_completed
from tqdm.auto import tqdm

# User provided helpers and constants
from helpers import get_idiom_name_by_folder, get_best_gpu
from constants import DATA_ROOT, FOLDER_NAMES

# %% [markdown]
# ## 0. Set Repository Path and Verify

# %%
# Define the path to your cloned omnilingual-asr repository
REPO_ROOT = Path("omnilingual-asr").resolve()  # <-- CHANGE THIS TO THE ACTUAL PATH
if not REPO_ROOT.exists():
    raise FileNotFoundError(f"Repository not found at {REPO_ROOT}. Please set the correct path.")

print(f"Using omnilingual-asr repository at: {REPO_ROOT}")

# Add repo to PYTHONPATH (ensures modules can be imported)
os.environ["PYTHONPATH"] = f"{REPO_ROOT}:{os.environ.get('PYTHONPATH', '')}"

# %% [markdown]
# ## 1. Hardware & Path Setup (unchanged)

# %%
best_gpu = get_best_gpu()
DEVICE = torch.device(f"cuda:{best_gpu}" if torch.cuda.is_available() else "cpu")

if torch.cuda.is_available():
    os.environ["CUDA_VISIBLE_DEVICES"] = str(best_gpu)

OUTPUT_DIR_ARTIFACTS = Path("./omnilingual-ctc-rm-all").absolute()
OUTPUT_DIR_ARTIFACTS.mkdir(parents=True, exist_ok=True)

print(f"Using GPU: {best_gpu}")
print(f"Artifacts will be saved to: {OUTPUT_DIR_ARTIFACTS}")

# %% [markdown]
# ## 2. Data Preparation (make dataset root absolute)

# %%
schema = pa.schema([
    ('text', pa.string()),
    ('audio_bytes', pa.list_(pa.int8())),
    ('audio_size', pa.int64()),
    ('corpus', pa.dictionary(pa.int32(), pa.string())),
    ('split', pa.dictionary(pa.int32(), pa.string())),
    ('language', pa.dictionary(pa.int32(), pa.string()))
])

# Use an absolute path for the dataset root
PARQUET_ROOT = Path("/local/scratch/matuor/dataset_root_dir/version=0").absolute()
PARQUET_ROOT.mkdir(parents=True, exist_ok=True)

def process_single_audio(audio_path, text, corpus_name, split_name, lang_code):
    if not os.path.exists(audio_path): return None
    try:
        wav, _ = librosa.load(audio_path, sr=16000, mono=True)
        buffer = io.BytesIO()
        sf.write(buffer, wav, 16000, format='ogg')
        raw_bytes = buffer.getvalue()
        int8_list = np.frombuffer(raw_bytes, dtype=np.int8).tolist()
        return {
            'text': text,
            'audio_bytes': int8_list,
            'audio_size': len(wav),
            'corpus': corpus_name,
            'split': split_name,
            'language': lang_code
        }
    except Exception as e:
        print(f"Error processing {audio_path}: {e}")
        return None

def process_partition(split_name, dataframe):
    records = []
    with ProcessPoolExecutor(max_workers=max(1, os.cpu_count()-1)) as ex:
        futures = [ex.submit(process_single_audio, r['f'], str(r['s']), "romansh", split_name, "roh_Latn") 
                   for _, r in dataframe.iterrows()]
        for f in tqdm(as_completed(futures), total=len(futures), desc=f"Prep {split_name}"):
            res = f.result()
            if res: records.append(res)
    
    if not records: return
    table = pa.Table.from_pandas(pd.DataFrame(records), schema=schema)
    out_dir = PARQUET_ROOT / f"corpus=romansh/split={split_name}/language=roh_Latn"
    out_dir.mkdir(parents=True, exist_ok=True)
    pq.write_table(table, out_dir / f"data_{len(list(out_dir.glob('*.parquet')))}.parquet")

# Process splits – uncomment and adapt as needed
# for folder in FOLDER_NAMES:
#     p = Path(DATA_ROOT) / folder
#     for s_in, s_out in {"train":"train", "validation":"dev", "test":"test"}.items():
#         tsv = p / f"{s_in}.tsv"
#         if tsv.exists():
#             df = pd.read_csv(tsv, sep='\t')
#             df['f'] = df['path'].apply(lambda x: str(p / "clips" / x))
#             df['s'] = df['sentence']
#             process_partition(s_out, df)

# %% [markdown]
# ## 3. Config Generation (with explicit Parquet settings)

# ## Compute Dataset Statistics

# %%
# Define paths (adjust if necessary)
parquet_root = PARQUET_ROOT  # This should be the root of your versioned dataset, e.g., /local/scratch/matuor/dataset_root_dir/version=0
stats_output = parquet_root.parent / "stats.tsv"  # Save stats one level above the version directory

print(f"Computing statistics for dataset at: {parquet_root}")
print(f"Stats will be saved to: {stats_output}")

cmd = [
    "python", "-m", "workflows.dataprep.hf_dataset_ingestion_example",
    "compute_stats",
    str(parquet_root),
    str(stats_output)
]

print("Running command:", " ".join(cmd))

# Run from the repository root (adjust cwd if needed)
import subprocess
result = subprocess.run(cmd, cwd=str(REPO_ROOT), capture_output=True, text=True)

if result.returncode != 0:
    print("❌ compute_stats failed!")
    print("STDERR:", result.stderr)
else:
    print("✅ compute_stats completed successfully.")
    print(result.stdout)

# %%
# Ensure the stats file exists (generated in step 4)
stats_file = PARQUET_ROOT / "stats.tsv"
if not stats_file.exists():
    raise FileNotFoundError(
        f"Statistics file not found at {stats_file}. "
        "Please run the compute_stats step first."
    )

# Asset Card (unchanged)
card_path = REPO_ROOT / "src" / "omnilingual_asr" / "cards" / "datasets" / "romansh_dataset.yaml"
card_path.parent.mkdir(parents=True, exist_ok=True)
with open(card_path, "w") as f:
    f.write(f"""\
name: romansh_dataset
dataset_family: mixture_parquet_asr_dataset
dataset_config:
  data: {PARQUET_ROOT}
tokenizer_ref: omniASR_tokenizer_v1
""")

# Training Recipe – now includes full Parquet configuration
config_path = REPO_ROOT / "workflows" / "recipes" / "wav2vec2" / "asr" / "configs" / "romansh_finetune.yaml"
config_path.parent.mkdir(parents=True, exist_ok=True)

config = {
    "dataset": {
        "name": "romansh_dataset",
        "train_split": "train",
        "valid_split": "dev",
        "storage_mode": "MIXTURE_PARQUET",                     # <-- CRITICAL
        "task_mode": "ASR",
        "mixture_parquet_storage_config": {                     # Required for parquet
            "dataset_summary_path": str(stats_file),
            "beta_corpus": 0.5,
            "beta_language": 0.5,
            "fragment_loading": {"cache": True}
        },
        "asr_task_config": {
            "max_audio_len": 960000,
            "max_num_elements": 7680000
        }
    },
    "optimizer": {
        "config": {"lr": 1e-05}
    },
    "trainer": {
        "grad_accumulation": {"num_batches": 4}
    },
    "regime": {
        "num_steps": 10000,
        "validate_every_n_steps": 500,
        "checkpoint_every_n_steps": 500
    }
}

with open(config_path, "w") as f:
    yaml.dump(config, f, default_flow_style=False)

print(f"Asset card written to {card_path}")
print(f"Training config written to {config_path}")

# %% [markdown]
# ## 4. Run Training from within the Repository

# %%
# The command as per the official README
cmd = [
    "python", "-m", "workflows.recipes.wav2vec2.asr",
    str(OUTPUT_DIR_ARTIFACTS),
    "--config-file", str(config_path)
]

print(f"🚀 Launching Training: {' '.join(cmd)}")
print(f"Working directory: {REPO_ROOT}")

# Run the command from the repo root
process = subprocess.Popen(
    cmd,
    cwd=str(REPO_ROOT),
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    env=os.environ.copy()
)

for line in process.stdout:
    print(line, end="")

process.wait()
if process.returncode != 0:
    print(f"❌ Training failed with exit code {process.returncode}")
else:
    print("✅ Training completed successfully!")

Using omnilingual-asr repository at: /local/scratch/matuor/omnilingual-asr
Selected GPU 7 with 24121 MiB free memory
Using GPU: 7
Artifacts will be saved to: /local/scratch/matuor/omnilingual-ctc-rm-all
Computing statistics for dataset at: /local/scratch/matuor/dataset_root_dir/version=0
Stats will be saved to: /local/scratch/matuor/dataset_root_dir/stats.tsv
Running command: python -m workflows.dataprep.hf_dataset_ingestion_example compute_stats /local/scratch/matuor/dataset_root_dir/version=0 /local/scratch/matuor/dataset_root_dir/stats.tsv


❌ compute_stats failed!
STDERR: Traceback (most recent call last):
  File "<frozen runpy>", line 198, in _run_module_as_main
  File "<frozen runpy>", line 88, in _run_code
  File "/local/scratch/matuor/omnilingual-asr/workflows/dataprep/hf_dataset_ingestion_example.py", line 18, in <module>
    from audio_tools import AudioTableProcessor, map_to_target_schema
ModuleNotFoundError: No module named 'audio_tools'



FileNotFoundError: Statistics file not found at /local/scratch/matuor/dataset_root_dir/version=0/stats.tsv. Please run the compute_stats step first.